# 103 - The Other Bases

While the Zernike polynomials are what you will use 99% of the time, prysm
supports a wide range of other orthogonal polynomial types:

- Jacobi
- Chebyshev
- Legendre
- Hermite
- Laguerre
- Dickson
- XY

They all have a similar API to the Zernikes, except they tend to have only
one indexing scheme.  The most common ones you will reach for are likely
the XY polynomials, Chebyshev, or Jacobi with α=β=0, which are the Legendre
polynomials.  You can think of the Jacobi family as orthogonal versions of 
rising powers, $1, x, x^2, x^3$, etc.  The Chebyshevs are closer to being
$\cos(0), \cos(x), \cos(2*x)$, etc.

This document will show each type of polynomial, as well as the pattern of the
API.  We'll start with the API pattern, then show a gallery of each of them.

----

All of the polynomial families are exported under `prysm.polynomials`, so  you
can browse them using tab completion in your editor or jupyter.  We'll use the
Jacobi polynomials as an example to show the API pattern:

```python
from prysm.polynomials import (
    jacobi,         # one mode
    jacobi_der,     # d/dx of that mode
    jacobi_seq,     # multiple modes (much faster)
    jacobi_der_seq  # d/dx of each mode
)
```

From here, we'll set up our usual grid and begin the gallery:

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from prysm.coordinates import make_xy_grid, cart_to_polar
from prysm.geometry import circle

# a 256x256 grid spanning the unit disk; r,t are polar; mask is True OUTSIDE
x, y = make_xy_grid(256, diameter=2)
r, t = cart_to_polar(x, y)
out = ~circle(1, r)        # boolean: True outside the unit circle (for blanking)

def show(arr, ax=None, blank=True, **kw):
    a = np.asarray(arr, dtype=float).copy()
    if blank:
        a[out] = np.nan
    ax = ax or plt.gca()
    return ax.imshow(a, cmap=kw.pop('cmap', 'RdBu'), **kw)

x1 = x[0, :]        # a 1D slice across the grid, x in [-1, 1]

## Jacobi: the parent family

The Jacobi polynomials are the core of prysm's polynomials module.  Most other
bases are actually computed by calling the Jacobi functions, because most other
bases are special cases of the Jacobi polynomials.

The Jacobi polynomials are orthogonal over $[-1,1]$, subject to the weighting
function controlled by the α,β  parameters.

In [ ]:
from prysm.polynomials import jacobi_seq

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x1, jacobi_seq([1, 2, 3, 4, 5], 0, 0, x1).T)
ax.set(title='Jacobi $P_n^{(0,0)}$ (= Legendre), orders 1-5', xlabel='x')

## Chebyshev: four kinds, for series and freeforms

The chebyshev polynomials come in four separate flavors,
`cheby1, cheby2, cheby3, cheby4`.  They are the same underlying
shapes, squished to the left or the right.

The Chebyshev polynomials are orthogonal over $[-1,1]$, subject to similar
weighting as the Jacobies.  The first kind are unweighted.  You may notice
the second kind look a lot like the unweighted Jacobi polynomials.

In [ ]:
from prysm.polynomials import cheby1, cheby2, cheby3, cheby4, cheby1_seq, cheby2_seq, cheby3_seq, cheby4_seq

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x1, cheby1_seq([1, 2, 3, 4, 5], x1).T)
ax.set(title='Chebyshev of the first kind $T_n$, orders 1-5', xlabel='x')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x1, cheby2_seq([1, 2, 3, 4, 5], x1).T)
ax.set(title='Chebyshev of the second kind $U_n$, orders 1-5', xlabel='x')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x1, cheby3_seq([1, 2, 3, 4, 5], x1).T)
ax.set(title='Chebyshev of the third kind $V_n$, orders 1-5', xlabel='x')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x1, cheby4_seq([1, 2, 3, 4, 5], x1).T)
ax.set(title='Chebyshev of the fourth kind $W_n$, orders 1-5', xlabel='x')

fig, ax = plt.subplots(figsize=(7, 4))
for f, lbl in [(cheby1, 'T (1st)'), (cheby2, 'U (2nd)'),
               (cheby3, 'V (3rd)'), (cheby4, 'W (4th)')]:
    ax.plot(x1, f(3, x1), label=lbl)
ax.set(title='Chebyshev, order 5, all four kinds', xlabel='x'); ax.legend()

## Legendre, Hermite, Laguerre

Three additional families are common, which have different orthogonal domains.

- Legendre - orthogonal on $[-1,1]$ with unit weight
- Hermite - orthogonal on $(-\infty, \infty)$ - both conventions (H and He) are supported
- Laguerre - orthogonal on $[0,\infty)$

In [ ]:
from prysm.polynomials import legendre_seq, hermite_He_seq, laguerre_seq

fig, axs = plt.subplots(1, 3, figsize=(13, 3.5))
axs[0].plot(x1, legendre_seq([1, 2, 3, 4, 5], x1).T); axs[0].set(title='Legendre', xlabel='x')
axs[1].plot(x1, hermite_He_seq([0, 1, 2, 3, 4], x1).T); axs[1].set(title="Hermite (He)", xlabel='x')
xl = np.linspace(0, 10, 256)
axs[2].plot(xl, laguerre_seq([0, 1, 2, 3, 4], alpha=0, x=xl).T); axs[2].set(title='Laguerre', xlabel='x')

## Dickson: monomials and Fibonacci polynomials

The Dickson polynomials are a lesser-known family with a tunable parameter
$\alpha$: $\alpha=0$ gives the plain monomials $x^n$, $\alpha=-1$ gives the
Fibonacci polynomials.

In [ ]:
from prysm.polynomials import dickson1_seq, dickson2_seq

fig, axs = plt.subplots(1, 2, figsize=(9, 3.5))
axs[0].plot(x1, dickson1_seq([1, 2, 3, 4, 5], 0, x1).T)
axs[0].set(title='Dickson (1st), alpha=0 -> monomials', xlabel='x')
axs[1].plot(x1, dickson2_seq([1, 2, 3, 4, 5], -1, x1).T)
axs[1].set(title='Dickson (2nd), alpha=-1 -> Fibonacci', xlabel='x')

## XY: 2D monomials for freeforms

When a surface has no symmetry, the simplest 2D basis is the polynomial $x^m y^n$.
The `xy` family indexes them with a single $j$ (`xy_j_to_mn`) and evaluates a whole
basis with `xy_seq(mns, x, y)`.  These are the literal terms a Zemax or Code V "XY
polynomial" freeform surface uses.

In [ ]:
from prysm.polynomials import xy_seq, xy_j_to_mn

mns = [xy_j_to_mn(j) for j in range(1, 10)]
basis_xy = xy_seq(mns, x, y)
m, n = xy_j_to_mn(8)
mode = basis_xy[7].copy(); mode[out] = np.nan
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(mode, cmap='RdBu'); ax.set(title=f'XY j=8  ->  $x^{m} y^{n}$', xticks=[], yticks=[]);

## Wrap-up

| family     | domain / weight        | reach for it when...                       |
|------------|------------------------|--------------------------------------------|
| Jacobi     | $[-1,1]$, tunable      | you want the parent / a custom weight      |
| Chebyshev  | $[-1,1]$, by kind      | series approximation, rectangular freeforms|
| Legendre   | $[-1,1]$, unit         | plain rectangular apertures                |
| Hermite    | line, Gaussian         | Hermite-Gauss laser modes                  |
| Laguerre   | $[0,\infty)$           | Laguerre-Gauss modes, semi-infinite domains|
| Dickson    | tunable ($\alpha$)     | monomials / Fibonacci without leaving OPs  |
| XY         | 2D monomials           | freeform surfaces with no symmetry         |

Every one shares the `f` / `f_seq` / `f_der` interface, so swapping bases is a
one-line change.  104 turns to a basis built for a specific job - the Forbes Q
polynomials for manufacturable aspheres - and the Clenshaw recurrence that sums
them stably.